In [1]:
from torch.utils.data import Dataset 
from torch.utils.data import DataLoader
import torch 
import os 
import numpy as np 

train_path: str = "../data/ServerMachineDataset/train"
test_path: str = "../data/ServerMachineDataset/test"
test_label_path: str = "../data/ServerMachineDataset/test_label"


class SMDDataset(Dataset): 

    def __init__(self, root: str, window_size: int = 100): 
        self.root = root 
        self.window_size = window_size
        self.records: list = []
        self.machine_count: int = len(os.listdir(self.root))

        machines: list[str] = os.listdir(self.root)
        for machine_id, machine_filename in enumerate(machines):
            machine_path: str = os.path.join(self.root, machine_filename)

            # read the file 
            machine_data: torch.Tensor = torch.from_numpy(
                np.loadtxt(machine_path, delimiter=",", dtype=np.float32), 
            )

            # split the data into windows
            for i in range(0, len(machine_data) - self.window_size + 1, self.window_size):
                window_data: torch.Tensor = machine_data[i:i+self.window_size]
                self.records.append((machine_id, window_data))

    def __len__(self) -> int:
        return len(self.records)
    
    def __getitem__(self, idx: int) -> tuple[int, torch.Tensor]:
        return self.records[idx]
    
    
train = SMDDataset(train_path, window_size=100)
test = SMDDataset(test_path, window_size=100)
 


In [25]:
loader = DataLoader(train, batch_size=32, shuffle=True)
machine_ids, sensor = next(iter(loader))
time_window, channels = sensor.shape[1:]
sensor.shape

torch.Size([32, 100, 38])

In [3]:
machine_ids.shape

torch.Size([32])

In [73]:
import torch.nn as nn 

embedding = nn.Embedding(num_embeddings=train.machine_count, embedding_dim=100)
machine_embeds = embedding(machine_ids)

In [ ]:
temporal_patterns = 20
kernel_size = 7
stride = 2

conv = nn.Conv1d(
    in_channels=channels, 
    out_channels=temporal_patterns, 
    kernel_size=7, 
    stride=2
)

transformed_window = (time_window - kernel_size) // stride + 1

trf_x = conv(sensor.transpose(1, 2)).transpose(1, 2)
cat_x  = torch.concat((
   trf_x[:, :, None, :].expand(-1, -1, transformed_window, -1), 
   trf_x[:, None, :, :].expand(-1, transformed_window, -1, -1)
), dim=-1)

w = nn.Linear(in_features=2*temporal_patterns, out_features=1, bias=False)
leaky_relu = nn.LeakyReLU(negative_slope=0.01)
alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
alpha = alpha.squeeze(-1)

In [ ]:
class TimeOrientedGAT(nn.Module): 
    def __init__(self, transformed_window: int, temporal_patterns: int, negative_slope: float = 0.01): 
        super(TimeOrientedGAT, self).__init__()
        self.transformed_window = transformed_window
        self.temporal_patterns = temporal_patterns
        self.negative_slope = negative_slope
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        cat_x  = torch.concat((
            x[:, :, None, :].expand(-1, -1, self.transformed_window, -1), 
            x[:, None, :, :].expand(-1, self.transformed_window, -1, -1)
        ), dim=-1)

        w = nn.Linear(in_features=2*self.temporal_patterns, out_features=1, bias=False)
        leaky_relu = nn.LeakyReLU(negative_slope=self.negative_slope)
        alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
        alpha = alpha.squeeze(-1)

        return alpha @ x 

In [54]:
tgat = TimeOrientedGAT(transformed_window=transformed_window, temporal_patterns=temporal_patterns)
tattn = tgat(trf_x)

In [71]:
class FeatureOrientedGAT(nn.Module): 
    def __init__(self, transformed_window: int, temporal_patterns: int, negative_slope: float = 0.01): 
        super(FeatureOrientedGAT, self).__init__()
        self.transformed_window = transformed_window
        self.temporal_patterns = temporal_patterns
        self.negative_slope = negative_slope
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feature_x = x.transpose(1, 2)
        cat_x  = torch.concat((
            feature_x[:, :, None, :].expand(-1, -1, self.temporal_patterns, -1), 
            feature_x[:, None, :, :].expand(-1, self.temporal_patterns, -1, -1)
        ), dim=-1)

        w = nn.Linear(in_features=2*self.transformed_window, out_features=1, bias=False)
        leaky_relu = nn.LeakyReLU(negative_slope=self.negative_slope)
        alpha = torch.softmax(leaky_relu(w(cat_x)), dim=2)
        alpha = alpha.squeeze(-1)

        return (alpha @ feature_x).transpose(1, 2)

fgat = FeatureOrientedGAT(transformed_window=transformed_window, temporal_patterns=temporal_patterns)
fattn = fgat(trf_x)
fattn


tensor([[[-7.8781e-04, -4.6873e-04, -1.2010e-03,  ..., -5.5846e-04,
          -5.6739e-04, -8.1896e-04],
         [-7.8164e-03, -7.4276e-03, -8.2884e-03,  ..., -7.5396e-03,
          -7.5504e-03, -7.8532e-03],
         [ 8.8508e-04,  1.2665e-03,  4.1467e-04,  ...,  1.1581e-03,
           1.1475e-03,  8.4828e-04],
         ...,
         [ 2.5738e-03,  2.7186e-03,  2.3813e-03,  ...,  2.6787e-03,
           2.6746e-03,  2.5591e-03],
         [ 2.7988e-03,  2.9119e-03,  2.6347e-03,  ...,  2.8822e-03,
           2.8790e-03,  2.7864e-03],
         [-4.1241e-03, -4.0063e-03, -4.3027e-03,  ..., -4.0385e-03,
          -4.0418e-03, -4.1379e-03]],

        [[-3.5751e-03, -3.6621e-03, -3.5472e-03,  ..., -3.6087e-03,
          -3.4506e-03, -3.5013e-03],
         [-8.8306e-03, -8.9065e-03, -8.8047e-03,  ..., -8.8621e-03,
          -8.7265e-03, -8.7628e-03],
         [ 9.4117e-04,  8.1926e-04,  9.8181e-04,  ...,  8.9123e-04,
           1.1306e-03,  1.0539e-03],
         ...,
         [-8.5829e-03, -8

In [ ]:
X = torch.concat((
    machine_embeds[:, None, :].expand(-1, transformed_window, -1), 
    tattn, 
    trf_x, 
    fattn, 
), dim=-1)

encoder = nn.GRU(
    input_size=3 * temporal_patterns + 100, 
    hidden_size=temporal_patterns, 
    num_layers=50, 
    batch_first=True
)

output, hn = encoder(X)

class Encoder(nn.Module): 
    def __init__(self, embedding_size: int, temporal_patterns: int, num_layers: int): 
        super(Encoder, self).__init__()
        self.gru = nn.GRU(
            input_size=3*temporal_patterns + embedding_size, 
            hidden_size=temporal_patterns, 
            num_layers=num_layers, 
            batch_first=True
        )
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        _, hn = self.gru(x)
        return hn  



(tensor([[[-0.0623,  0.0844, -0.0423,  ..., -0.0173,  0.0369,  0.0273],
          [-0.1073,  0.1259, -0.0492,  ..., -0.0231,  0.0652,  0.0471],
          [-0.1332,  0.1414, -0.0435,  ..., -0.0234,  0.0859,  0.0635],
          ...,
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641],
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641],
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641]],
 
         [[-0.0623,  0.0844, -0.0423,  ..., -0.0173,  0.0369,  0.0273],
          [-0.1073,  0.1259, -0.0492,  ..., -0.0231,  0.0652,  0.0471],
          [-0.1332,  0.1414, -0.0435,  ..., -0.0234,  0.0859,  0.0635],
          ...,
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641],
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641],
          [-0.1379,  0.1599, -0.0342,  ..., -0.0048,  0.1385,  0.0641]],
 
         [[-0.0623,  0.0844, -0.0423,  ..., -0.0173,  0.0369,  0.0273],
          [-0.1073,  0.1259,

In [76]:
machine_embeds.shape

torch.Size([32, 100])

In [77]:
tattn.shape

torch.Size([32, 47, 20])